# Large Language Model
## MiniGPT
This project implements a transformer-based large language model (LLM) capable of generating text by predicting tokens in sequence. It uses attention mechanisms to capture long-range dependencies and learns linguistic patterns from large-scale text datasets. The training pipeline leverages PyTorch for defining the model architecture and performing GPU-accelerated tensor operations, while the FastTokenizer (Byte-Level BPE) converts raw text into numerical token IDs that the model can process efficiently. AdamW is used as the optimizer, providing adaptive learning rates and improved regularization for stable convergence. Hyperparameters such as learning rate, batch size, context window, and model depth control the behavior, stability, and performance of the model. The training loop utilizes DataLoader for efficient batching and shuffling, and tqdm for real-time progress visualization. The recorded training metrics show a subtle but consistent diminishing loss across epochs for both training and validation sets, indicating that the model is converging, generalizing well, and improving its ability to predict the next token. This combination of stable, aligned, and gradually decreasing losses confirms that the model is learning effectively and that the overall architecture, preprocessing, and optimization strategy are successfully producing a high-quality language model.

In [1]:
# Basic imports
from os import path, fsync
import csv
import os
import math
from re import sub
from datetime import datetime

# Third Party imports
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm


# Hugging Face
from datasets import load_dataset
from datasets import Dataset as HFDataset
from tokenizers import ByteLevelBPETokenizer
from transformers import PreTrainedTokenizerFast

TRAINING_MODE = True

# Check for GPU. Use if available. Otherwise, you are going to have a bad time.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


### Load a small subset of wikipedia, clean it to remove headings and other unhelpful formats, and return a cleaned dataset
While a larger dataset (ds) would produce a better LLM, time a cost is a factor, and this is fine for a proof of concept.

In [2]:
# - Load a small subset (10%) of the Wikipedia dataset
# - Using the '20231101.en' configuration (English Wikipedia snapshot)
ds = load_dataset("wikimedia/wikipedia", "20231101.en", split="train[:10%]")
assert isinstance(ds, HFDataset)

# Step 2: Define a text-cleaning function
def clean_text(example):
    """
    Cleans a single dataset example.

    1. Remove Wikipedia-style headings (== Heading ==)
    2. Remove text inside brackets (usually references or citations)
    3. Normalize whitespace (multiple spaces -> single space)

    This ensures the text is cleaner for language modeling or NLP tasks.
    """
    text = example["text"]

    # Remove headings like '== Heading =='
    text = sub(r"={2,}.*?={2,}", "", text)

    # Remove references or other bracketed content
    text = sub(r"\[.*?\]", "", text)

    # Replace multiple whitespaces/newlines with a single space
    text = sub(r"\s+", " ", text).strip()

    return {"text": text}

# Step 3: Apply cleaning to the dataset
# - We remove unnecessary columns ('id', 'url', 'title') because they are not needed
# - The map function applies the cleaning function to every row efficiently
clean_ds = ds.map(clean_text, remove_columns=["id", "url", "title"])

# Step 4: Filter out very short entries
# - Very short entries may not be useful for language modeling
# - Filter to only keep entries longer than 200 characters
clean_ds = clean_ds.filter(lambda x: len(x["text"]) > 200)

# Step 5: Print the number of remaining cleaned entries
print("Number of cleaned entries:", len(clean_ds)) # pyright: ignore[reportArgumentType]


Number of cleaned entries: 624928


In [ ]:
# Step 1: Load an existing Byte-Level BPE tokenizer
# - vocab.json: contains the token vocabulary
# - merges.txt: contains BPE merge rules
# Using an existing tokenizer avoids retraining from scratch
tokenizer = ByteLevelBPETokenizer(
    "tokenizer/vocab.json",
    "tokenizer/merges.txt"
)

# Step 2: Wrap the tokenizer into Hugging Face's PreTrainedTokenizerFast
# - Provides full compatibility with HF Transformers library
# - Allows usage of special tokens like BOS, EOS, and UNK
tok = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,   # trained tokenizer object
    bos_token="<|bos|>",          # beginning-of-sequence token
    eos_token="<|eos|>",          # end-of-sequence token
    unk_token="<|unk|>"           # unknown token for OOV words
)

# Step 3: Save the tokenizer in a format compatible with Hugging Face
# - Can be reloaded easily with `from_pretrained`
tok.save_pretrained("tokenizer_hf")

print("Tokenizer ready!")

In [ ]:
# if path.exists("train.pt"): print("Encoded dataset already exists. Skipping encoding step.")

# Step 0: Define end-of-text token
EOT = "<|eos|>"
# Step 1: Set batch size for processing
#  - Process data in chunks to avoid memory issues
batch_size = 1000

# Step 2: Prepare a list to store token IDs
ids_list = []

# Step 3: Loop over the cleaned dataset in batches
clean_ids = tqdm(range(0, len(clean_ds), batch_size),
                 desc="Encoding dataset",
                 leave = True,
                 position=1)
for start in clean_ids:
    # Select a batch of examples
    batch = clean_ds.select(range(start, min(start + batch_size, len(clean_ds))))

    # Extract the raw text from each example
    texts = batch["text"]

    # Step 4: Encode text using the tokenizer
    # - Append the end-of-text token to each example
    # - Returns a dictionary, input_ids contains the token IDs
    encoded = tok([text + EOT for text in texts])["input_ids"]
    # Step 5: Convert each encoded sequence to a PyTorch tensor and append to list
    for e in encoded:
        ids_list.append(torch.tensor(e, dtype=torch.long))

# Step 6: Concatenate all sequences into a single 1D tensor
# - This makes it easy to create training sequences later
ids = torch.cat(ids_list)

# Step 7: Save the encoded dataset to disk
torch.save(ids, "train.pt")
print("Encoded dataset saved. Total tokens:", len(ids))


In [ ]:
# Character-level dataset for next-token prediction
class CharDataset(Dataset):
    def __init__(self, ids, block_size=1024):
        # Store the full list of token IDs
        self.ids = ids
        # Length of each training block (sequence length)
        self.block_size = block_size
        # Number of full blocks available
        self.n_blocks = len(ids) // block_size

    def __len__(self):
        # Returns the number of training samples
        return self.n_blocks

    def __getitem__(self, idx):
        # Calculate start/end positions of the block
        start = idx * self.block_size
        end = start + self.block_size
        # Input sequence x
        x = self.ids[start:end]
        # Target y: shifted by one token for next-token prediction
        y = self.ids[start+1:end+1]
        return x, y

# Define block size
block_size = 1024
# Initialize dataset
dataset = CharDataset(ids, block_size=block_size)
# Create initial DataLoader
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# --- Dataset splitting for validation ---
# Percentage of dataset to use for validation
val_split = 0.1
val_size = int(len(dataset) * val_split)
# Training size is total minus validation size
train_size = len(dataset) - val_size
# Split dataset into train and validation subsets
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# --- Recreate DataLoaders ---
# Updated batch size for training
batch_size = 10
# Training DataLoader
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# Validation DataLoader (no shuffle for reproducibility)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Print sizes for verification
print("DataLoaders recreated. Train samples:", len(train_dataset), "Val samples:", len(val_dataset))


In [ ]:
# MiniGPT: a simplified Transformer-based language model
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, n_embd=128, n_layer=4, block_size=1024):
        super().__init__()
        # Token embedding: converts token IDs into dense vectors
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        # Positional embedding: provides each token with a position encoding
        self.pos_emb = nn.Embedding(block_size, n_embd)

        # Transformer encoder stack
        # Each layer has self-attention and feed-forward components
        # batch_first=True so inputs are read as (batch, seq, embd). Without it, PyTorch
        # treats dim 0 as the sequence, so attention ran across samples in the batch
        # instead of across tokens within each sequence.
        # ModuleList (not Sequential) so each block can receive the causal mask.
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=n_embd,   # embedding size
                nhead=4,          # number of attention heads
                batch_first=True  # inputs are (batch, seq, embd)
            ) for _ in range(n_layer)
        ])

        # Final layer normalization to stabilize training
        self.ln_f = nn.LayerNorm(n_embd)

        # Output projection to vocabulary logits
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

        # Maximum sequence length
        self.block_size = block_size

    def forward(self, idx):
        # idx shape: (batch_size, sequence_length)
        b, t = idx.size()

        # Token embeddings
        tok_embeddings = self.tok_emb(idx)  # (b, t, n_embd)

        # Generate positions for the sequence
        positions = torch.arange(t, device=idx.device).unsqueeze(0)
        # Positional embeddings
        pos_embeddings = self.pos_emb(positions)  # (1, t, n_embd)

        # Add token + positional embeddings
        x = tok_embeddings + pos_embeddings

        # Causal mask: each position may only attend to itself and earlier positions,
        # so the model cannot see the token it is being trained to predict
        mask = nn.Transformer.generate_square_subsequent_mask(t, device=idx.device)

        # Transformer encoder layers
        for block in self.blocks:
            x = block(x, src_mask=mask, is_causal=True)  # (b, t, n_embd)

        # Layer normalization
        x = self.ln_f(x)

        # Final logits over vocabulary
        logits = self.head(x)  # (b, t, vocab_size)
        return logits

# Instantiate the model with chosen hyperparameters
# Hyperparameters: These are configuration settings chosen BEFORE training starts.
# They control the model's structure and behavior (e.g., size, depth, sequence length).
# Unlike model weights, hyperparameters are not learned — they define how learning happens.
# vocab_size: total number of unique tokens the model can generate
# n_embd: size of each token/positional embedding vector (higher = more representational power)
# n_layer: number of transformer encoder layers stacked (controls model depth)
# block_size: the maximum sequence length the model supports
vocab_size = tok.vocab_size
model = MiniGPT(
    vocab_size=vocab_size,
    n_embd=128,      # embedding dimension
    n_layer=4,       # number of transformer layers
    block_size=block_size  # max input sequence length
).to(device)


### Note: retrain from scratch after the attention fix
Earlier runs used `batch_first=False` with no causal mask, so attention mixed tokens across different samples in a batch rather than within each sequence. Checkpoints and loss history from those runs do not carry over: delete `checkpoint.pt` and archive the old `training_log.csv` before training again.

In [ ]:
if TRAINING_MODE:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    tok = PreTrainedTokenizerFast.from_pretrained("tokenizer_hf")

    # # Recreate architecture
    # model = MiniGPT(
    #     vocab_size=tok.vocab_size,
    #     n_embd=128,
    #     n_layer=4,
    #     block_size=1024
    # ).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=3e-4)
    criterion = nn.CrossEntropyLoss()

    start_epoch = 0

    # Load checkpoint if exists
    if path.exists("checkpoint.pt"):
        ckpt = torch.load("checkpoint.pt", map_location=device)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        start_epoch = ckpt["epoch"] + 1
        print(f"Loaded checkpoint — resuming from epoch {start_epoch}")
    else:
        print("No checkpoint found — starting fresh")

In [ ]:
if TRAINING_MODE:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    def log_training_progress(csv_file, epoch, avg_train_loss, avg_val_loss, total_tokens):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        file_exists = path.exists(csv_file)

        with open(csv_file, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)

            # write header only once
            if not file_exists:
                writer.writerow(["Timestamp", "Epoch", "Train Loss", "Val Loss", "Total Tokens"])

            writer.writerow([timestamp, epoch + 1, avg_train_loss, avg_val_loss, total_tokens])

            f.flush()
            fsync(f.fileno())

    def save_checkpoint(model,
                        optimizer,
                        epoch,
                        checkpoint_path="checkpoint.pt"
                        ):
        # Write to a temp file first, then swap it in, so an interrupted save
        # can never corrupt the last good checkpoint
        tmp_path = checkpoint_path + ".tmp"
        torch.save({
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "epoch": epoch
        }, tmp_path)
        os.replace(tmp_path, checkpoint_path)

    # --- Hyperparameters ---
    batch_size = 10
    block_size = 1024
    epochs = 1
    val_split = 0.1
    SEED = 42
    total_tokens = len(dataset.ids)  # total number of token is the number of sequences)

    # --- Dataset split ---
    # Fixed SEED so the validation set is the same in every session
    val_size = int(len(dataset) * val_split)
    train_size = len(dataset) - val_size
    train_dataset, val_dataset = random_split(
        dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(SEED)
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # --- Model ---
    # The optimizer and criterion come from the checkpoint cell. Recreating the
    # optimizer here would throw away the state loaded from the checkpoint.
    model.to(device)

    # --- Training loop ---
    for epoch in range(start_epoch, start_epoch + epochs):
        # Training
        model.train()
        train_loss = 0
        train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]", leave=True, position=0)

        for x_batch, y_batch in train_loop:
            x_batch = x_batch.to(device).clamp(0, vocab_size - 1)
            y_batch = y_batch.to(device).clamp(0, vocab_size - 1)

            optimizer.zero_grad()
            logits = model(x_batch)
            loss = criterion(logits.view(-1, logits.size(-1)), y_batch.view(-1))
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_loop.set_postfix(loss=train_loss / (train_loop.n + 1))

        avg_train_loss = train_loss / len(train_loader)

        # Validation
        model.eval()
        val_loss = 0
        val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]",
                        leave=True, position=1,
                        miniters=100)
        with torch.no_grad():
            for x_batch, y_batch in val_loop:
                x_batch = x_batch.to(device).clamp(0, vocab_size - 1)
                y_batch = y_batch.to(device).clamp(0, vocab_size - 1)

                logits = model(x_batch)
                loss = criterion(logits.view(-1, logits.size(-1)), y_batch.view(-1))
                val_loss += loss.item()
                val_loop.set_postfix(loss=val_loss / (val_loop.n + 1))

        avg_val_loss = val_loss / len(val_loader)

        # Save the checkpoint before logging, so the log never lists an epoch
        # that has no checkpoint behind it
        save_checkpoint(model, optimizer, epoch)
        print("Saved checkpoint!")

        log_training_progress("training_log.csv", epoch, avg_train_loss, avg_val_loss, total_tokens)
        print(f"Epoch {epoch+1} finished | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

In [ ]:
def generate(model, tokenizer, prompt, max_new_tokens=50, temperature=0.8, top_k=50,):

    """
    Generates text from a given prompt using an autoregressive language model.

    Args:
        model: The trained language model used for text generation. Must output logits for each token.
        tokenizer: The tokenizer used to convert between text and token IDs.
        prompt (str): The initial text to start generation from.
        max_new_tokens (int, optional): Maximum number of new tokens to generate. Default is 50.
        temperature (float, optional): Controls randomness of sampling; lower values make output more deterministic. Default is 0.8.
        top_k (int, optional): Limits sampling to the top-k most likely tokens to avoid unlikely outputs. Default is 50.

    Process:
        1. Converts the input prompt into token IDs and ensures all IDs are within the model's vocabulary range.
        2. Iteratively generates one token at a time:
            - Feeds the current token sequence into the model.
            - Applies temperature scaling and top-k filtering to the logits.
            - Samples the next token based on the resulting probability distribution.
            - Appends the new token to the sequence.
        3. Stops after generating `max_new_tokens` tokens.

    Returns:
        str: The original prompt followed by the generated text continuation.
    """
    model.eval()
    device = next(model.parameters()).device

    # Convert prompt to input IDs and clamp to valid range
    input_ids = tokenizer(prompt)["input_ids"]
    vocab_size = model.head.out_features
    unk_id = tokenizer.unk_token_id if tokenizer.unk_token_id is not None else 0
    input_ids = [i if 0 <= i < vocab_size else unk_id for i in input_ids]

    idx = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)
    top_k = min(top_k, vocab_size)

    for _ in range(max_new_tokens):
        with torch.no_grad():
            # Crop to the last block_size tokens so positional embeddings stay in range
            idx_cond = idx[:, -model.block_size:]
            logits = model(idx_cond)[:, -1, :]  # (1, vocab_size)
            logits = logits / temperature

            # Top-k filtering
            top_logits, top_indices = torch.topk(logits, top_k, dim=-1)
            probs = torch.softmax(top_logits, dim=-1)

            sampled = torch.multinomial(probs, num_samples=1)
            next_id = top_indices.gather(-1, sampled)

            idx = torch.cat([idx, next_id], dim=1)

    # Slice output after the prompt length
    prompt_len = len(input_ids)
    generated_ids = idx[0][prompt_len:].tolist()

    response = tokenizer.decode(generated_ids)

    return f"{prompt}\n{response}"


In [ ]:
PROMPT = "Who wrote Moby Dick?"

tok = PreTrainedTokenizerFast.from_pretrained("tokenizer_hf/")

TEMPERATURE=.8          # High temperature for more randomness
TOP_K=50                # High top-k for diversity
LEN_RESPONSE = 50       # number of tokens to generate

with open("training_log.csv", "r", encoding="utf-8") as f:
    lines = f.readlines()
    first_line = lines[1]
    last_line = lines[-1]
    prev_line = lines[-2]
    epoch = int(last_line.split(",")[1])
    train_loss_start = round(float(first_line.split(",")[2]), 5)
    train_loss_curr = round(float(last_line.split(",")[2]), 5)
    train_loss_prev = round(float(prev_line.split(",")[2]), 5)

# model takes ~40 miutes to train for 1 epoch on 10% of wikipedia dataset
estimated_training_hours = (40 / 60) * epoch
# get the name of the device trained on
trained_on = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"

print(f"""Generating response...\n
      Parameters:
      | epochs={epoch}
      | current training loss={train_loss_curr}
      | previous training loss={train_loss_prev}
      | starting training loss={train_loss_start}
      | estimated training hours={estimated_training_hours}
      | trained on={trained_on}
      | temperature={TEMPERATURE}
      | top k={TOP_K}
      | total tokens={len(ids)}\n"""
)
print(generate(model, tok, PROMPT, max_new_tokens=LEN_RESPONSE, temperature=TEMPERATURE, top_k=TOP_K))

In [ ]:
LOG_FILE = "training_log.csv"
BLUE, GRAY = "#0071e3", "#86868b"

epochs, train_loss, val_loss = [], [], []
with open(LOG_FILE, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        epochs.append(int(row["Epoch"]))
        train_loss.append(float(row["Train Loss"]))
        val_loss.append(float(row["Val Loss"]))

train_ppl = [math.exp(l) for l in train_loss]
val_ppl = [math.exp(l) for l in val_loss]
gain = (1 - val_ppl[-1] / val_ppl[0]) * 100

fig, ax = plt.subplots(figsize=(8, 5.5))

ax.plot(epochs, train_ppl, marker="o", linewidth=2, color=GRAY, label="Train")
ax.plot(epochs, val_ppl, marker="o", linewidth=3, color=BLUE, label="Validation")
ax.yaxis.set_major_formatter("{x:,.0f}")
ax.set_xticks(epochs)
ax.margins(y=0.12)

ax.annotate(f"{val_ppl[0]:,.0f}", (epochs[0], val_ppl[0]), xytext=(0, 12),
            textcoords="offset points", ha="center", color=BLUE, fontsize=11)
ax.annotate(f"{val_ppl[-1]:,.0f}", (epochs[-1], val_ppl[-1]), xytext=(0, 12),
            textcoords="offset points", ha="center", color=BLUE, fontsize=13, fontweight="bold")

ax.set_title(f"{gain:.1f}% lower perplexity since epoch 1", loc="left", fontsize=15, fontweight="bold")
ax.text(0.99, 0.02, "Y-axis zoomed", transform=ax.transAxes, ha="right", fontsize=9, color=GRAY)

ax.set_xlabel("Epoch")
ax.set_ylabel("Perplexity (lower is better)")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.2)
ax.legend(frameon=False)

plt.tight_layout()
plt.savefig("training_perplexity.png", dpi=150)
plt.show()